# 04. Chuẩn bị dữ liệu và xây dựng đặc trưng

Áp dụng các quy tắc làm sạch rồi tạo hai bộ đặc trưng: A gồm 127 biến hồ sơ hiện tại; B gồm 199 biến, bổ sung lịch sử đủ điều kiện quan sát. Các phép tổng hợp được đưa về một dòng cho mỗi `SK_ID_CURR`.

Biến số còn thiếu được giữ dưới dạng NaN. Trong notebook 06, LR/DT/RF học median và cờ thiếu trong từng fold; CatBoost/XGBoost giữ NaN số theo cách xử lý của thư viện. Bước chọn tập biến được trình bày riêng ở notebook 05.



## Mục lục

- [01. Đọc dữ liệu và xác nhận phạm vi](#part-1)
- [02. Chốt quy tắc chuẩn bị dữ liệu](#part-2)
- [03. Xây dựng bộ A: hồ sơ vay hiện tại](#part-3)
- [04. Tổng hợp sáu nguồn lịch sử](#part-4)
- [05. Ghép và hoàn thiện bộ B](#part-5)
- [06. Lập từ điển đặc trưng](#part-6)
- [07. Chuẩn bị pipeline tiền xử lý](#part-7)
- [08. Đọc đầu ra và đối chiếu ý nghĩa đặc trưng](#part-8)
- [09. Xuất kết quả và chốt bước chuẩn bị dữ liệu](#part-9)

<a id="muc-01"></a>

<a id="part-1"></a>

## 01. Đọc dữ liệu và xác nhận phạm vi
lấy hồ sơ và ID phân tập từ MySQL; tách TARGET khỏi dữ liệu tạo đặc trưng.
Các bảng lịch sử lớn được tổng hợp lần lượt trong `HomeCreditPreparation`. Notebook trình bày quy tắc, công thức và kết quả ở cấp hồ sơ.

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from sklearn.base import clone

pd.set_option('display.max_columns', 12)
pd.set_option('display.max_rows', 20)

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from connectors.Connector import Connector
from models.HomeCreditPreparation import HomeCreditPreparation

FEATURES = ROOT / 'data/features'
HISTORY = FEATURES / 'history'
REPORTS = ROOT / 'reports/data_preparation'
for folder in [FEATURES, HISTORY, REPORTS]:
    folder.mkdir(parents=True, exist_ok=True)


In [2]:
connector = Connector()
connector.connect()


In [3]:
preparation = HomeCreditPreparation(connector)
application = preparation.loadInputs()
split = preparation.split
feature_index = preparation.feature_index
train_ids = preparation.train_ids


In [4]:
saved_split = pd.read_parquet(ROOT / 'data/splits/application_split.parquet')
# SQL và Parquet có thể biểu diễn cột chữ bằng hai dtype khác nhau.
split = split.astype(saved_split.dtypes.to_dict())
if not split.equals(saved_split):
    raise ValueError('Phân tập khác notebook 01. Dừng trước khi tạo đặc trưng.')
if 'TARGET' in application.columns:
    raise ValueError('TARGET không được nằm trong dữ liệu tạo đặc trưng.')

split_counts = feature_index.groupby('split', sort=False).size()
display(split_counts.to_frame('n'))


,n
split,
application_test,48744
train,184506
validation,61502
test,61503


Có 356.255 hồ sơ: 184.506 train, 61.502 validation, 61.503 test nội bộ và 48.744 application_test không nhãn. Split/CV giữ nguyên từ notebook 01. TARGET không được đọc vào bảng đầu vào; ID chỉ dùng nối và truy vết, không đưa vào X. Các công thức cố định được áp dụng riêng cho từng hồ sơ, không học từ phân phối validation/test.

<a id="muc-02"></a>

<a id="part-2"></a>

## 02. Chốt quy tắc chuẩn bị dữ liệu

<a id="muc-02-1"></a>

### 02.1. Thời điểm thông tin và phạm vi lịch sử
xác định điều kiện sử dụng cho từng nguồn và các trường chưa đưa vào mô hình. Tách thời điểm sự kiện/cập nhật thực tế khỏi lịch dự kiến; ghi nhận giới hạn của mốc thời gian tương đối.

| Nguồn | Điều kiện dùng trong phiên bản này | Giới hạn |
|---|---|---|
| application | Thông tin hồ sơ hiện tại theo từ điển | Giả định bộ dữ liệu phản ánh thông tin đầu vào lúc chấm; chưa có log thời điểm từng trường |
| bureau | DAYS_CREDIT <= 0, DAYS_CREDIT_UPDATE <= 0 và UPDATE >= CREDIT | Trạng thái tại lần cập nhật gần nhất, không khẳng định dư nợ chính xác từng giây tại t0 |
| bureau_balance | Xác định chủ từ bureau đủ điều kiện; MONTHS_BALANCE < 0 | Không dùng dòng thiếu bảng cha; tháng 0 loại thận trọng, không gọi mọi tháng 0 là leakage |
| previous_application | DAYS_DECISION < 0; bỏ các ngày diễn biến thực tế chưa xác minh | Không lọc máy móc lịch dự kiến tương lai; phiên bản này chỉ chọn các trường hồ sơ/quyết định |
| POS/credit_card | MONTHS_BALANCE < 0, xác định được chủ hồ sơ | DPD là snapshot; tháng 0 của POS được mô tả tại application nhưng chưa đưa vào phiên bản này |
| installments | DAYS_ENTRY_PAYMENT <= 0 và DAYS_INSTALMENT <= 0; có khóa/ngày/tiền đủ điều kiện | Chỉ các khoản thanh toán đã quan sát cho kỳ đến hạn; chưa có lịch nghĩa vụ đầy đủ để suy ra mọi khoản chưa trả |

Các trường hợp bị loại vì thời điểm tương lai chỉ được đếm trong báo cáo phạm vi, không tạo thành đặc trưng dự báo. Quy tắc này thận trọng với dữ liệu có độ phân giải ngày/tháng; không chứng minh chính xác thứ tự trong ngày.

In [ ]:
time_rules = {
    'bureau': 'DAYS_CREDIT <= 0 and DAYS_CREDIT_UPDATE <= 0 and DAYS_CREDIT_UPDATE >= DAYS_CREDIT',
    'bureau_balance': 'Eligible bureau parent and MONTHS_BALANCE < 0',
    'previous_application': 'DAYS_DECISION < 0',
    'pos_cash_balance': 'MONTHS_BALANCE < 0',
    'credit_card_balance': 'MONTHS_BALANCE < 0',
    'installments_payments': 'DAYS_ENTRY_PAYMENT <= 0 and DAYS_INSTALMENT <= 0; valid keys and amounts',
}


Sáu nguồn có điều kiện quan sát riêng. Những trường chưa rõ thời điểm không được âm thầm đưa vào bộ B. Các ngày dự kiến tương lai chưa bị xem là lỗi chỉ vì mang dấu dương; danh sách trường được chọn trong từng mục dưới đây mới quyết định nội dung sử dụng.

<a id="muc-02-2"></a>

### 02.2. Giá trị đặc biệt và dữ liệu thiếu
chốt cách biểu diễn thiếu trước khi xây đặc trưng, đồng thời giữ phần học tham số cho pipeline trong fold.

| Trường hợp | Cách thực hiện |
|---|---|
| DAYS_EMPLOYED = 365243 | Tạo cờ riêng, đổi thành NaN trước khi quy đổi năm |
| OWN_CAR_AGE khi không có xe | Giữ trạng thái không áp dụng; cờ riêng với trường hợp có xe nhưng thiếu tuổi xe |
| Mã phân loại XNA/XAP hoặc mã khác | Giữ nguyên mã có trong từ điển; không gộp tất cả thành một loại thiếu |
| Ô phân loại thực sự thiếu | Điền hằng __MISSING__, không học mode trên toàn dữ liệu |
| Ô số bị thiếu | Giữ NaN trong A/B; median và chỉ báo thiếu được học trong fold train |
| Không có lịch sử đủ điều kiện | Cờ HAS_USABLE_* và count = 0; các thống kê tiền/thời gian vẫn NaN |

Thiếu theo điều kiện, không có lịch sử sử dụng được và ô thiếu được biểu diễn khác nhau. Không điền median/mode từ toàn bộ các tập, cũng không gán chắc MCAR/MAR/MNAR để biện minh cho một phép điền.

<a id="muc-02-3"></a>

### 02.3. Ngoại lai và trường hợp cần xác minh
chuyển kết quả kiểm tra của notebook 02 thành cách xử lý cụ thể, không xóa thông tin chỉ vì có giá trị cực đoan.

Giữ các khoản tiền cực đoan và số dư/dư nợ âm trong các biến mô tả được chọn. Không áp hàng rào IQR làm ngưỡng loại hồ sơ. Riêng tỷ số hoặc trạng thái có điều kiện hợp lệ thì để NaN khi mẫu số không dương hoặc thành phần chưa diễn giải được. Bureau có nhiều mã tiền tệ: các thống kê tiền chỉ dùng **currency 1** trong phiên bản này, các mã khác vẫn được đếm riêng; không cộng/trung bình tiền giữa các mã chưa có tỷ giá.

Bộ dữ liệu chưa bị clipping hoặc xóa hồ sơ cực đoan. Các đặc trưng vẫn cần được mô hình đánh giá bằng CV; giá trị lớn, tỷ số vượt 1 hoặc cờ thống kê chưa phải bằng chứng lỗi hay rủi ro. Mã currency 1 không được tự coi là đồng tiền hoặc đơn vị của application. Ở bước huấn luyện, nhóm sẽ so sánh LR hiện tại với một nhánh log1p cho bốn biến tiền không âm đã chọn, trước scaling và trong cùng ngân sách CV. Giữ nguyên dữ liệu ở đây chưa chứng minh đây là phương án tối ưu cho mọi mô hình.

<a id="muc-02-4"></a>

### 02.4. Khóa, mã định danh và kiểu dữ liệu
xác nhận cột phân loại/cột số từ schema; giữ mã hồ sơ làm index, không dùng mã định danh, TARGET, split hoặc fold làm biến giải thích.

In [6]:
raw_categorical = application.select_dtypes(include=['object', 'string', 'category']).columns
raw_numeric = application.columns.difference(raw_categorical)
column_types = pd.DataFrame({
    'type': ['numeric', 'categorical'],
    'columns_n': [len(raw_numeric), len(raw_categorical)]
})
display(column_types)


,type,columns_n
0,numeric,104
1,categorical,16


Mã hồ sơ nằm ngoài các cột đặc trưng. Những cột số hoặc phân loại giữ theo cấu trúc nguồn; schema cuối ghi lại đúng các cột đã xây dựng cho A/B; kiểu và thứ tự cột được lưu để các bước huấn luyện sử dụng nhất quán.

<a id="muc-03"></a>

<a id="part-3"></a>

## 03. Xây dựng bộ A: hồ sơ vay hiện tại

<a id="muc-03-1"></a>

### 03.1. Các trường hồ sơ hiện tại
áp dụng các quy tắc cố định cho thông tin hiện tại, tạo cờ sentinel/không áp dụng và quy đổi các trường ngày sang năm; ghi rõ số hồ sơ train chịu ảnh hưởng.

In [7]:
application = preparation.prepareApplication(application)

application_rule_counts = application.loc[train_ids,
    ['EMPLOYED_SENTINEL', 'CAR_AGE_NOT_APPLICABLE', 'CAR_AGE_MISSING_WITH_CAR']].sum().rename('train_n')
display(application_rule_counts.to_frame())


,train_n
EMPLOYED_SENTINEL,33122
CAR_AGE_NOT_APPLICABLE,121925
CAR_AGE_MISSING_WITH_CAR,4


Trên train có 33.122 hồ sơ mang sentinel thâm niên, 121.925 hồ sơ không có xe và 4 hồ sơ có xe nhưng thiếu tuổi xe. Các cờ được giữ riêng; năm trường ngày được thay bằng năm theo công thức cố định, chưa chuẩn hóa có học tham số. Mã phân loại có sẵn giữ nguyên; __MISSING__ chỉ thay ô thiếu thực sự.

<a id="muc-03-2"></a>

### 03.2. Các đặc trưng dẫn xuất từ hồ sơ
tạo một số tỷ số kỹ thuật đơn giản từ các khoản tiền và quy mô gia đình, bảo đảm mẫu số hợp lệ; lưu bộ A trước imputation/scaling có học.

Các tỷ số dưới đây dùng giá trị đã ghi trong dataset. Chưa đủ căn cứ coi là DTI chuẩn, thu nhập ròng khả dụng, kỳ hạn vay hoặc EAD.

In [ ]:
application = preparation.addApplicationRatios(application)

In [9]:
day_to_year = preparation.day_to_year
derived_definitions = [
    ['EMPLOYED_SENTINEL', 'DAYS_EMPLOYED', '1 if DAYS_EMPLOYED == 365243', 'Cờ giá trị đặc biệt thâm niên', 'cờ'],
    ['CAR_AGE_NOT_APPLICABLE', 'FLAG_OWN_CAR', '1 if FLAG_OWN_CAR == N', 'Tuổi xe không áp dụng theo cờ sở hữu', 'cờ'],
    ['CAR_AGE_MISSING_WITH_CAR', 'FLAG_OWN_CAR,OWN_CAR_AGE', 'FLAG_OWN_CAR == Y and OWN_CAR_AGE is missing', 'Có xe nhưng thiếu tuổi xe', 'cờ'],
    ['RATIO_CREDIT_INCOME_RECORDED', 'AMT_CREDIT,AMT_INCOME_TOTAL', 'AMT_CREDIT / AMT_INCOME_TOTAL if denominator > 0', 'Tỷ số hai giá trị đã ghi, không phải DTI', 'tỷ số kỹ thuật'],
    ['RATIO_ANNUITY_INCOME_RECORDED', 'AMT_ANNUITY,AMT_INCOME_TOTAL', 'AMT_ANNUITY / AMT_INCOME_TOTAL if denominator > 0', 'Tỷ số annuity/thu nhập chưa đồng bộ kỳ đo', 'tỷ số kỹ thuật'],
    ['RATIO_CREDIT_GOODS_RECORDED', 'AMT_CREDIT,AMT_GOODS_PRICE', 'AMT_CREDIT / AMT_GOODS_PRICE if denominator > 0', 'Tỷ số số tiền vay/giá hàng đã ghi', 'tỷ số kỹ thuật'],
    ['INCOME_DIV_FAMILY_SIZE_RECORDED', 'AMT_INCOME_TOTAL,CNT_FAM_MEMBERS', 'AMT_INCOME_TOTAL / CNT_FAM_MEMBERS if denominator > 0', 'Thu nhập khách hàng chia quy mô gia đình, không phải thu nhập hộ đã xác minh', 'đơn vị tiền nguồn/người']
]
for source, feature in day_to_year.items():
    derived_definitions.append([feature, source, '-source / 365.25 if source <= 0; employment sentinel -> NaN',
                                'Khoảng thời gian theo cột gốc: ' + source, 'năm'])
    
derived = pd.DataFrame(derived_definitions, columns=['feature', 'source_column', 'formula', 'meaning', 'unit'])

derived['source_table'] = 'application'
derived['feature_sets'] = 'A,B'
derived['time_rule'] = 'Current application information; retrospective availability assumption'
derived['limitation'] = 'Không phải chỉ tiêu tài chính chuẩn; không suy ra nhân quả hoặc tính hợp lệ từ giá trị đơn lẻ'
display(derived[['feature', 'formula', 'meaning']])


,feature,formula,meaning
0,EMPLOYED_SENTINEL,1 if DAYS_EMPLOYED == 365243,Cờ giá trị đặc biệt thâm niên
1,CAR_AGE_NOT_APPLICABLE,1 if FLAG_OWN_CAR == N,Tuổi xe không áp dụng theo cờ sở hữu
2,CAR_AGE_MISSING_WITH_CAR,FLAG_OWN_CAR == Y and OWN_CAR_AGE is missing,Có xe nhưng thiếu tuổi xe
3,RATIO_CREDIT_INCOME_RECORDED,AMT_CREDIT / AMT_INCOME_TOTAL if denominator > 0,"Tỷ số hai giá trị đã ghi, không phải DTI"
4,RATIO_ANNUITY_INCOME_RECORDED,AMT_ANNUITY / AMT_INCOME_TOTAL if denominator > 0,Tỷ số annuity/thu nhập chưa đồng bộ kỳ đo
5,RATIO_CREDIT_GOODS_RECORDED,AMT_CREDIT / AMT_GOODS_PRICE if denominator > 0,Tỷ số số tiền vay/giá hàng đã ghi
6,INCOME_DIV_FAMILY_SIZE_RECORDED,AMT_INCOME_TOTAL / CNT_FAM_MEMBERS if denomina...,"Thu nhập khách hàng chia quy mô gia đình, khôn..."
7,AGE_YEARS,-source / 365.25 if source <= 0; employment se...,Khoảng thời gian theo cột gốc: DAYS_BIRTH
8,EMPLOYED_YEARS,-source / 365.25 if source <= 0; employment se...,Khoảng thời gian theo cột gốc: DAYS_EMPLOYED
9,REGISTRATION_YEARS,-source / 365.25 if source <= 0; employment se...,Khoảng thời gian theo cột gốc: DAYS_REGISTRATION


**Xem phân phối các biến dẫn xuất trên train**

In [ ]:
feature_columns_a = application.columns.tolist()
display(application.loc[train_ids, derived['feature']].describe().T[['count', 'min', '50%', 'max']])
print('Số đặc trưng A:', len(feature_columns_a))

,count,min,50%,max
EMPLOYED_SENTINEL,184506.0,0.000000,0.000000,1.000000e+00
CAR_AGE_NOT_APPLICABLE,184506.0,0.000000,1.000000,1.000000e+00
CAR_AGE_MISSING_WITH_CAR,184506.0,0.000000,0.000000,1.000000e+00
RATIO_CREDIT_INCOME_RECORDED,184506.0,0.037500,3.267273,4.922720e+01
RATIO_ANNUITY_INCOME_RECORDED,184498.0,0.003333,0.163000,1.570600e+00
RATIO_CREDIT_GOODS_RECORDED,184352.0,0.150000,1.118800,6.000000e+00
INCOME_DIV_FAMILY_SIZE_RECORDED,184504.0,5142.857143,75000.000000,4.500022e+06
AGE_YEARS,184506.0,20.503765,43.104723,6.907324e+01
EMPLOYED_YEARS,151384.0,-0.000000,4.517454,4.904038e+01
REGISTRATION_YEARS,184506.0,-0.000000,12.320329,6.754825e+01


**Lưu riêng bộ A theo từng tập**

In [11]:
partition_map = feature_index.set_index('SK_ID_CURR')['split']
(FEATURES / 'a').mkdir(exist_ok=True)
for partition in ['train', 'validation', 'test', 'application_test']:
    selected_ids = partition_map.index[(partition_map == partition)]
    application.loc[selected_ids].reset_index().to_parquet(FEATURES / 'a' / (partition + '.parquet'), index=False)

del application


Bộ A có 127 đặc trưng, gồm 120 trường nguồn sau thay đơn vị ngày, 3 cờ và 4 tỷ số dẫn xuất. Bốn tập được lưu riêng, giữ NaN số để xử lý trong fold train; mẫu số thiếu/không dương cho kết quả NaN. Thống kê hiển thị chỉ dùng train; TARGET không tham gia công thức tạo các biến dẫn xuất.

<a id="muc-04"></a>

<a id="part-4"></a>

## 04. Tổng hợp sáu nguồn lịch sử

Mỗi bảng được xử lý riêng về cấp hồ sơ. Cờ có lịch sử ở bộ B chỉ phản ánh **có bản ghi đủ điều kiện trong nguồn đang có**, không khẳng định người vay lần đầu hay lịch sử tốt/xấu. Từng biến và điều kiện thời gian là thiết kế của đồ án.

<a id="muc-04-1"></a>

### 04.1. Lịch sử tín dụng: bureau
sử dụng trạng thái ở lần cập nhật đủ điều kiện, tổng hợp số khoản và các thống kê tín dụng. Giữ các khoản khác currency 1 trong biến đếm, không gộp tiền khác mã khi chưa có tỷ giá.

SQL trong `processBureau()` chọn các cột cần dùng, lọc ngày tín dụng/cập nhật và nối danh sách hồ sơ. `Connector.queryDataset(sql)` trả DataFrame; phép tổng hợp tiền theo currency 1 giữ trong pandas để bảo toàn kết quả số thực.

In [ ]:
aggregate = preparation.processBureau()
train_aggregate = aggregate.loc[aggregate.index.intersection(train_ids)]
display(train_aggregate.head())

,BUR_LOAN_N,BUR_ACTIVE_N,BUR_CLOSED_N,BUR_OTHER_CURRENCY_N,BUR_C1_CREDIT_MEAN,BUR_C1_CREDIT_MAX,...,BUR_C1_OVERDUE_MEAN,BUR_C1_OVERDUE_MAX,BUR_C1_ANNUITY_MEAN,BUR_OVERDUE_DAYS_MAX,BUR_CREDIT_DAYS_AGO_MEAN,BUR_RECENT_UPDATE_DAYS_AGO
SK_ID_CURR,,,,,,,,,,,,,
215354,11,6,5,0,5.430859e+05,2700000.00,...,0.0,0.0,NaN,0,727.818182,16
162297,6,3,3,0,1.371731e+06,7033500.00,...,0.0,0.0,NaN,0,1344.500000,31
222183,8,5,3,0,8.948700e+05,3709552.50,...,0.0,0.0,NaN,0,959.125000,24
426155,4,0,4,0,8.213886e+04,182791.44,...,0.0,0.0,NaN,0,1309.750000,337
136226,2,1,1,0,1.113750e+05,112500.00,...,0.0,0.0,NaN,0,540.000000,30


In [13]:
bureau_features = pd.DataFrame([
    ['BUR_LOAN_N', 'SK_ID_BUREAU', 'size', 'Số khoản tín dụng có bản ghi đủ điều kiện', 'khoản'],
    ['BUR_ACTIVE_N', 'ACTIVE_RECORD', 'sum', 'Số khoản có trạng thái Active ở lần cập nhật', 'khoản'],
    ['BUR_CLOSED_N', 'CLOSED_RECORD', 'sum', 'Số khoản có trạng thái Closed ở lần cập nhật', 'khoản'],
    ['BUR_OTHER_CURRENCY_N', 'OTHER_CURRENCY', 'sum', 'Số khoản có mã tiền khác currency 1', 'khoản'],
    ['BUR_C1_CREDIT_MEAN', 'AMT_CREDIT_SUM', 'mean', 'Trung bình tiền tín dụng chỉ trong currency 1', 'đơn vị nguồn currency 1'],
    ['BUR_C1_CREDIT_MAX', 'AMT_CREDIT_SUM', 'max', 'Tiền tín dụng lớn nhất trong currency 1', 'đơn vị nguồn currency 1'],
    ['BUR_C1_DEBT_MEAN', 'AMT_CREDIT_SUM_DEBT', 'mean', 'Trung bình dư nợ báo cáo trong currency 1, giữ số âm', 'đơn vị nguồn currency 1'],
    ['BUR_C1_DEBT_MAX', 'AMT_CREDIT_SUM_DEBT', 'max', 'Dư nợ báo cáo lớn nhất trong currency 1', 'đơn vị nguồn currency 1'],
    ['BUR_C1_OVERDUE_MEAN', 'AMT_CREDIT_SUM_OVERDUE', 'mean', 'Trung bình tiền quá hạn báo cáo trong currency 1', 'đơn vị nguồn currency 1'],
    ['BUR_C1_OVERDUE_MAX', 'AMT_CREDIT_SUM_OVERDUE', 'max', 'Tiền quá hạn báo cáo lớn nhất trong currency 1', 'đơn vị nguồn currency 1'],
    ['BUR_C1_ANNUITY_MEAN', 'AMT_ANNUITY', 'mean', 'Trung bình annuity báo cáo trong currency 1', 'đơn vị/kỳ nguồn chưa xác minh'],
    ['BUR_OVERDUE_DAYS_MAX', 'CREDIT_DAY_OVERDUE', 'max', 'Ngày quá hạn lớn nhất theo bản ghi bureau', 'ngày'],
    ['BUR_CREDIT_DAYS_AGO_MEAN', 'CREDIT_DAYS_AGO', 'mean', 'Trung bình số ngày từ khoản tín dụng tới hồ sơ', 'ngày'],
    ['BUR_RECENT_UPDATE_DAYS_AGO', 'UPDATE_DAYS_AGO', 'min', 'Số ngày từ lần cập nhật gần nhất đủ điều kiện', 'ngày']
], columns=['feature', 'source_column', 'aggregation', 'meaning', 'unit'])

bureau_features['formula'] = bureau_features['aggregation'] + '(' + bureau_features['source_column'] + ') per SK_ID_CURR; money only currency 1'
bureau_features['source_table'] = 'bureau'
bureau_features['feature_sets'] = 'B'
bureau_features['time_rule'] = time_rules['bureau']
bureau_features['limitation'] = 'Trạng thái ở lần cập nhật; currency 1 chưa xác minh tương đương tiền application; mã đếm theo flags trong code'
bureau_features = bureau_features.drop(columns='aggregation')
display(bureau_features[['feature', 'meaning', 'unit']])


,feature,meaning,unit
0,BUR_LOAN_N,Số khoản tín dụng có bản ghi đủ điều kiện,khoản
1,BUR_ACTIVE_N,Số khoản có trạng thái Active ở lần cập nhật,khoản
2,BUR_CLOSED_N,Số khoản có trạng thái Closed ở lần cập nhật,khoản
3,BUR_OTHER_CURRENCY_N,Số khoản có mã tiền khác currency 1,khoản
4,BUR_C1_CREDIT_MEAN,Trung bình tiền tín dụng chỉ trong currency 1,đơn vị nguồn currency 1
5,BUR_C1_CREDIT_MAX,Tiền tín dụng lớn nhất trong currency 1,đơn vị nguồn currency 1
6,BUR_C1_DEBT_MEAN,"Trung bình dư nợ báo cáo trong currency 1, giữ...",đơn vị nguồn currency 1
7,BUR_C1_DEBT_MAX,Dư nợ báo cáo lớn nhất trong currency 1,đơn vị nguồn currency 1
8,BUR_C1_OVERDUE_MEAN,Trung bình tiền quá hạn báo cáo trong currency 1,đơn vị nguồn currency 1
9,BUR_C1_OVERDUE_MAX,Tiền quá hạn báo cáo lớn nhất trong currency 1,đơn vị nguồn currency 1


In [14]:
aggregate.to_parquet(HISTORY / 'bureau.parquet')
del aggregate, train_aggregate


Bureau giữ 1.716.276/1.716.428 dòng đủ điều kiện, trong đó 879.778 dòng thuộc train; tổng hợp về một dòng/hồ sơ. Các mã tiền khác vẫn được đếm nhưng không tham gia thống kê tiền currency 1. Số dư âm được giữ; trạng thái báo cáo không được diễn giải thành TARGET của hồ sơ hiện tại.

<a id="muc-04-2"></a>

### 04.2. Trạng thái tín dụng theo tháng: bureau_balance
xác định chủ từ bureau đủ điều kiện, tổng hợp snapshot theo khoản tín dụng rồi về hồ sơ. Phân biệt STATUS số, C đóng và X chưa biết; mẫu số tỷ lệ tháng có dấu hiệu DPD chỉ gồm các tháng có mã 0–5.

SQL trong `processBureauBalance()` lọc MONTHS_BALANCE < 0 và đọc theo lô qua Connector. Chủ khoản được xác định từ bureau đủ điều kiện đã giữ trong phiên; thống kê C/X và DPD giữ đúng công thức đã chốt. Bản thử SQL tổng hợp toàn bộ được lưu riêng, chưa dùng thay thế.

In [ ]:
aggregate = preparation.processBureauBalance(preparation.bureau_owners)
train_aggregate = aggregate.loc[aggregate.index.intersection(train_ids)]
display(train_aggregate.head())

,BB_LOAN_N,BB_MONTH_N,BB_KNOWN_DPD_MONTH_N,BB_DPD_MONTH_N,BB_CLOSED_MONTH_N,BB_UNKNOWN_MONTH_N,BB_MAX_STATUS_CODE,BB_OLDEST_MONTH,BB_RECENT_MONTH,BB_DPD_MONTH_SHARE
SK_ID_CURR,,,,,,,,,,
380361,17,386,105,0,126,155,0.0,-86,-1,0.000000
401081,4,124,11,0,101,12,0.0,-85,-1,0.000000
448798,2,37,10,0,27,0,0.0,-27,-1,0.000000
370938,6,166,116,21,46,4,2.0,-49,-1,0.181034
400009,7,260,54,0,199,7,0.0,-91,-1,0.000000


In [16]:
bureau_balance_features = pd.DataFrame([
    ['BB_LOAN_N', 'SK_ID_BUREAU', 'size', 'Số khoản có ít nhất một snapshot đủ điều kiện', 'khoản'],
    ['BB_MONTH_N', 'MONTH_N', 'sum', 'Số snapshot khoản-tháng được quan sát', 'snapshot'],
    ['BB_KNOWN_DPD_MONTH_N', 'KNOWN_DPD_N', 'sum', 'Số snapshot có STATUS 0–5', 'snapshot'],
    ['BB_DPD_MONTH_N', 'DPD_N', 'sum', 'Số snapshot có STATUS 1–5', 'snapshot'],
    ['BB_CLOSED_MONTH_N', 'CLOSED_N', 'sum', 'Số snapshot có STATUS C', 'snapshot'],
    ['BB_UNKNOWN_MONTH_N', 'UNKNOWN_N', 'sum', 'Số snapshot có STATUS X', 'snapshot'],
    ['BB_MAX_STATUS_CODE', 'MAX_CODE', 'max', 'Mã STATUS số lớn nhất quan sát, không phải số ngày DPD', 'mã 0–5'],
    ['BB_OLDEST_MONTH', 'OLDEST_MONTH', 'min', 'Tháng tương đối xa nhất có snapshot', 'tháng tương đối'],
    ['BB_RECENT_MONTH', 'RECENT_MONTH', 'max', 'Tháng tương đối gần nhất có snapshot', 'tháng tương đối']
], columns=['feature', 'source_column', 'aggregation', 'meaning', 'unit'])

bureau_balance_features['formula'] = bureau_balance_features['aggregation'] + '(' + bureau_balance_features['source_column'] + ') across eligible loans; monthly flags defined above'
bureau_balance_features['source_table'] = 'bureau_balance -> eligible bureau'
bureau_balance_features['feature_sets'] = 'B'
bureau_balance_features['time_rule'] = time_rules['bureau_balance']
bureau_balance_features['limitation'] = 'Snapshot khoản-tháng không phải kỳ thanh toán; STATUS 5 gồm 120+ hoặc sold/written off; C/X không là thang DPD'
extra = bureau_balance_features.iloc[[0]].copy()
extra[['feature', 'source_column', 'formula', 'meaning', 'unit']] = [
    'BB_DPD_MONTH_SHARE', 'STATUS', 'sum(STATUS in 1..5) / sum(STATUS in 0..5)',
    'Tỷ lệ snapshot có mã 1–5 trong snapshot có mã DPD biết được', 'tỷ lệ']
bureau_balance_features = pd.concat([bureau_balance_features, extra], ignore_index=True)
bureau_balance_features = bureau_balance_features.drop(columns='aggregation')
display(bureau_balance_features[['feature', 'meaning', 'unit']])


,feature,meaning,unit
0,BB_LOAN_N,Số khoản có ít nhất một snapshot đủ điều kiện,khoản
1,BB_MONTH_N,Số snapshot khoản-tháng được quan sát,snapshot
2,BB_KNOWN_DPD_MONTH_N,Số snapshot có STATUS 0–5,snapshot
3,BB_DPD_MONTH_N,Số snapshot có STATUS 1–5,snapshot
4,BB_CLOSED_MONTH_N,Số snapshot có STATUS C,snapshot
5,BB_UNKNOWN_MONTH_N,Số snapshot có STATUS X,snapshot
6,BB_MAX_STATUS_CODE,"Mã STATUS số lớn nhất quan sát, không phải số ...",mã 0–5
7,BB_OLDEST_MONTH,Tháng tương đối xa nhất có snapshot,tháng tương đối
8,BB_RECENT_MONTH,Tháng tương đối gần nhất có snapshot,tháng tương đối
9,BB_DPD_MONTH_SHARE,Tỷ lệ snapshot có mã 1–5 trong snapshot có mã ...,tỷ lệ


In [17]:
aggregate.to_parquet(HISTORY / 'bureau_balance.parquet')
del aggregate, train_aggregate


Có 23.590.930/27.299.925 snapshot đạt đồng thời điều kiện tháng và bảng cha; 8.663.845 snapshot thuộc train. Dữ liệu được tổng hợp qua cấp khoản rồi về hồ sơ. Tỷ lệ DPD chỉ dùng STATUS 0–5 trong mẫu số; C/X được đếm riêng, không ép thành số quá hạn.

<a id="muc-04-3"></a>

### 04.3. Hồ sơ vay trước: previous_application
tổng hợp hồ sơ có quyết định trước lúc chấm, tách các trạng thái quyết định và các khoản tiền đã ghi. Không dùng các ngày diễn biến thực tế hoặc trường có sentinel chưa cần thiết cho bộ biến được chọn.

`HomeCreditPreparation.processPrevious()` viết rõ SQL lọc hồ sơ quá khứ và nối danh sách hồ sơ; `Connector.queryDataset(sql)` trả DataFrame. Các phép tổng hợp pandas giữ nguyên để kết quả số thực khớp bộ đặc trưng đã chốt.

In [18]:
aggregate = preparation.processPrevious()
train_aggregate = aggregate.loc[aggregate.index.intersection(train_ids)]
display(train_aggregate.head())


,PREV_APPLICATION_N,PREV_APPROVED_N,PREV_REFUSED_N,PREV_REFUSED_SHARE,PREV_REQUESTED_MEAN,PREV_APPROVED_CREDIT_MEAN,PREV_APPROVED_CREDIT_MAX,PREV_APPROVED_ANNUITY_MEAN,PREV_APPROVED_PAYMENT_COUNT_MEAN,PREV_RECENT_DECISION_DAYS_AGO
SK_ID_CURR,,,,,,,,,,
108129,6,6,0,0.000000,201525.750000,257522.25,679671.0,13039.485000,15.666667,164
176158,23,6,15,0.652174,259704.997826,374201.25,470790.0,37802.220000,12.666667,218
202054,25,8,13,0.520000,179037.720000,141430.50,744498.0,10501.959375,11.375000,7
175704,4,3,0,0.000000,86560.875000,110472.00,135000.0,8826.090000,7.333333,14
296299,17,5,4,0.235294,122146.676471,207525.60,512370.0,12416.706000,12.400000,21


In [19]:
previous_features = pd.DataFrame([
    ['PREV_APPLICATION_N', 'SK_ID_PREV', 'size', 'Số hồ sơ trước có quyết định < 0', 'hồ sơ'],
    ['PREV_APPROVED_N', 'APPROVED_RECORD', 'sum', 'Số hồ sơ có trạng thái Approved', 'hồ sơ'],
    ['PREV_REFUSED_N', 'REFUSED_RECORD', 'sum', 'Số hồ sơ có trạng thái Refused', 'hồ sơ'],
    ['PREV_REFUSED_SHARE', 'REFUSED_RECORD', 'mean', 'Tỷ lệ hồ sơ Refused trong hồ sơ đủ điều kiện', 'tỷ lệ'],
    ['PREV_REQUESTED_MEAN', 'AMT_APPLICATION', 'mean', 'Trung bình số tiền từng đề nghị', 'đơn vị tiền nguồn'],
    ['PREV_APPROVED_CREDIT_MEAN', 'APPROVED_AMT_CREDIT', 'mean', 'Trung bình số tiền tín dụng của hồ sơ Approved', 'đơn vị tiền nguồn'],
    ['PREV_APPROVED_CREDIT_MAX', 'APPROVED_AMT_CREDIT', 'max', 'Số tiền tín dụng lớn nhất của hồ sơ Approved', 'đơn vị tiền nguồn'],
    ['PREV_APPROVED_ANNUITY_MEAN', 'APPROVED_AMT_ANNUITY', 'mean', 'Trung bình annuity của hồ sơ Approved', 'đơn vị/kỳ nguồn chưa xác minh'],
    ['PREV_APPROVED_PAYMENT_COUNT_MEAN', 'APPROVED_CNT_PAYMENT', 'mean', 'Trung bình số kỳ trả theo hồ sơ Approved', 'kỳ theo nguồn'],
    ['PREV_RECENT_DECISION_DAYS_AGO', 'DECISION_DAYS_AGO', 'min', 'Ngày từ quyết định gần nhất đủ điều kiện', 'ngày']
], columns=['feature', 'source_column', 'aggregation', 'meaning', 'unit'])

previous_features['formula'] = previous_features['aggregation'] + '(' + previous_features['source_column'] + ') per SK_ID_CURR; APPROVED_* masks status Approved'
previous_features['source_table'] = 'previous_application'
previous_features['feature_sets'] = 'B'
previous_features['time_rule'] = 'DAYS_DECISION < 0; no unverified actual lifecycle dates'
previous_features['limitation'] = 'Hồ sơ Approved không tự chứng minh đã giải ngân/trả nợ tốt; kỳ đo tiền chưa xác minh đầy đủ'
previous_features = previous_features.drop(columns='aggregation')
display(previous_features[['feature', 'meaning', 'unit']])


,feature,meaning,unit
0,PREV_APPLICATION_N,Số hồ sơ trước có quyết định < 0,hồ sơ
1,PREV_APPROVED_N,Số hồ sơ có trạng thái Approved,hồ sơ
2,PREV_REFUSED_N,Số hồ sơ có trạng thái Refused,hồ sơ
3,PREV_REFUSED_SHARE,Tỷ lệ hồ sơ Refused trong hồ sơ đủ điều kiện,tỷ lệ
4,PREV_REQUESTED_MEAN,Trung bình số tiền từng đề nghị,đơn vị tiền nguồn
5,PREV_APPROVED_CREDIT_MEAN,Trung bình số tiền tín dụng của hồ sơ Approved,đơn vị tiền nguồn
6,PREV_APPROVED_CREDIT_MAX,Số tiền tín dụng lớn nhất của hồ sơ Approved,đơn vị tiền nguồn
7,PREV_APPROVED_ANNUITY_MEAN,Trung bình annuity của hồ sơ Approved,đơn vị/kỳ nguồn chưa xác minh
8,PREV_APPROVED_PAYMENT_COUNT_MEAN,Trung bình số kỳ trả theo hồ sơ Approved,kỳ theo nguồn
9,PREV_RECENT_DECISION_DAYS_AGO,Ngày từ quyết định gần nhất đủ điều kiện,ngày


In [20]:
aggregate.to_parquet(HISTORY / 'previous_application.parquet')
del aggregate, train_aggregate


1.670.214 hồ sơ trước đạt điều kiện quyết định, trong đó 848.478 thuộc train. Thống kê tiền Approved được tính riêng. Các trường ngày có sentinel không nằm trong bộ biến được chọn nên không đưa 365243 vào mean/max ngày. Approved chưa đồng nghĩa đã sử dụng vốn hoặc trả nợ tốt.

<a id="muc-04-4"></a>

### 04.4. Lịch sử trả góp theo tháng: POS_CASH_balance
tổng hợp số khoản, snapshot, trạng thái DPD và số kỳ còn lại được báo cáo ở những tháng trước application. Mẫu số tỷ lệ DPD chỉ gồm snapshot có giá trị DPD hợp lệ.

SQL trong `HomeCreditPreparation.processPos()` dùng `WHERE`, `JOIN` và `GROUP BY` để tổng hợp tại MySQL. `Connector.queryDataset(sql)` trả về một dòng cho mỗi hồ sơ; các mục sau dùng lại kết quả này.

In [21]:
aggregate = preparation.processPos()
train_aggregate = aggregate.loc[aggregate.index.intersection(train_ids)]
display(train_aggregate.head())


,POS_LOAN_N,POS_MONTH_N,POS_KNOWN_DPD_MONTH_N,POS_DPD_MONTH_SHARE,POS_DPD_MAX,POS_DPD_DEF_MAX,POS_FUTURE_INST_MEAN,POS_OLDEST_MONTH,POS_RECENT_MONTH
SK_ID_CURR,,,,,,,,,
182943,2,43,43,0.000000,0,0,25.395349,-62,-2
367990,2,27,27,0.000000,0,0,19.111111,-83,-16
397406,8,109,109,0.174312,485,6,11.743119,-94,-1
269225,6,114,114,0.000000,0,0,21.692982,-87,-1
334279,7,126,126,0.000000,0,0,14.476190,-89,-1


In [22]:
pos_features = pd.DataFrame([
    ['POS_LOAN_N', 'SK_ID_PREV', 'nunique', 'Số khoản có snapshot đủ điều kiện', 'khoản'],
    ['POS_MONTH_N', 'MONTHS_BALANCE', 'size', 'Số snapshot khoản-tháng đủ điều kiện', 'snapshot'],
    ['POS_KNOWN_DPD_MONTH_N', 'SK_DPD', 'count', 'Số snapshot có DPD hợp lệ', 'snapshot'],
    ['POS_DPD_MONTH_SHARE', 'OBS_DPD_FLAG', 'mean', 'Tỷ lệ snapshot DPD > 0 trong snapshot có DPD hợp lệ', 'tỷ lệ'],
    ['POS_DPD_MAX', 'SK_DPD', 'max', 'DPD lớn nhất được báo cáo tại snapshot', 'ngày'],
    ['POS_DPD_DEF_MAX', 'SK_DPD_DEF', 'max', 'DPD_DEF lớn nhất theo quy tắc nguồn', 'ngày'],
    ['POS_FUTURE_INST_MEAN', 'CNT_INSTALMENT_FUTURE', 'mean', 'Trung bình số kỳ còn lại được báo cáo ở các snapshot', 'kỳ theo nguồn'],
    ['POS_OLDEST_MONTH', 'MONTHS_BALANCE', 'min', 'Tháng tương đối xa nhất có snapshot', 'tháng tương đối'],
    ['POS_RECENT_MONTH', 'MONTHS_BALANCE', 'max', 'Tháng tương đối gần nhất có snapshot', 'tháng tương đối']
], columns=['feature', 'source_column', 'aggregation', 'meaning', 'unit'])

pos_features['formula'] = pos_features['aggregation'] + '(' + pos_features['source_column'] + ') per SK_ID_CURR; OBS_DPD_FLAG = SK_DPD > 0 if known'
pos_features['source_table'] = 'pos_cash_balance'
pos_features['feature_sets'] = 'B'
pos_features['time_rule'] = 'MONTHS_BALANCE < 0; known nonnegative DPD for DPD summaries'
pos_features['limitation'] = 'Snapshot không phải kỳ thanh toán; tỷ lệ bị ảnh hưởng độ dài/mức bao phủ lịch sử'
pos_features = pos_features.drop(columns='aggregation')
display(pos_features[['feature', 'meaning', 'unit']])


,feature,meaning,unit
0,POS_LOAN_N,Số khoản có snapshot đủ điều kiện,khoản
1,POS_MONTH_N,Số snapshot khoản-tháng đủ điều kiện,snapshot
2,POS_KNOWN_DPD_MONTH_N,Số snapshot có DPD hợp lệ,snapshot
3,POS_DPD_MONTH_SHARE,Tỷ lệ snapshot DPD > 0 trong snapshot có DPD h...,tỷ lệ
4,POS_DPD_MAX,DPD lớn nhất được báo cáo tại snapshot,ngày
5,POS_DPD_DEF_MAX,DPD_DEF lớn nhất theo quy tắc nguồn,ngày
6,POS_FUTURE_INST_MEAN,Trung bình số kỳ còn lại được báo cáo ở các sn...,kỳ theo nguồn
7,POS_OLDEST_MONTH,Tháng tương đối xa nhất có snapshot,tháng tương đối
8,POS_RECENT_MONTH,Tháng tương đối gần nhất có snapshot,tháng tương đối


In [23]:
aggregate.to_parquet(HISTORY / 'pos_cash_balance.parquet')
del aggregate, train_aggregate


Có 10.001.358 snapshot POS đủ điều kiện, trong đó 5.125.192 thuộc train. Số khoản, số snapshot và tỷ lệ DPD có đơn vị khác nhau; các biến phản ánh lịch sử được quan sát, chưa phải số lần vỡ nợ hoặc số kỳ trả nợ độc lập.

<a id="muc-04-5"></a>

### 04.5. Lịch sử thẻ tín dụng: credit_card_balance
tổng hợp tiền và DPD ở các snapshot trước application; tính tỷ số số dư/hạn mức trên cùng khoản và cùng tháng với mẫu số dương. Giữ số dư âm và các tỷ số vượt 1 để tiếp tục đánh giá.

SQL trong `processCard()` lọc tháng quá khứ và nối danh sách hồ sơ. `Connector.queryBatches(sql)` đọc theo lô; pandas tính các thống kê và tỷ số với mẫu số dương, giữ số dư âm và các giá trị thiếu.

In [24]:
aggregate = preparation.processCard()
train_aggregate = aggregate.loc[aggregate.index.intersection(train_ids)]
display(train_aggregate.head())


,CC_LOAN_N,CC_MONTH_N,CC_KNOWN_DPD_MONTH_N,CC_DPD_MONTH_SHARE,CC_DPD_MAX,CC_BALANCE_MEAN,...,CC_LIMIT_MEAN,CC_BALANCE_LIMIT_RATIO_MEAN,CC_BALANCE_LIMIT_RATIO_MAX,CC_PAYMENT_TOTAL_MEAN,CC_DRAWINGS_MEAN,CC_RECENT_MONTH
SK_ID_CURR,,,,,,,,,,,,,
363914,1,96,96,0.010417,1,32895.901875,...,63750.0,0.533325,1.458459,5061.5625,3732.434062,-1
337855,1,15,15,0.000000,0,189173.943000,...,201000.0,0.899210,1.057263,8783.6700,17323.389000,-1
367360,1,7,7,0.000000,0,117935.382857,...,292500.0,0.403198,0.996728,90420.1650,81005.348571,-1
340339,1,10,10,0.000000,0,82206.657000,...,110250.0,0.672206,1.085303,3465.5895,12554.964000,-1
289902,1,12,12,0.000000,0,109440.476250,...,112500.0,0.962222,1.125762,9834.9375,15071.250000,-1


In [25]:
card_features = pd.DataFrame([
    ['CC_LOAN_N', 'SK_ID_PREV', 'nunique', 'Số tài khoản có snapshot đủ điều kiện', 'tài khoản'],
    ['CC_MONTH_N', 'MONTHS_BALANCE', 'size', 'Số snapshot tài khoản-tháng', 'snapshot'],
    ['CC_KNOWN_DPD_MONTH_N', 'SK_DPD', 'count', 'Số snapshot có DPD hợp lệ', 'snapshot'],
    ['CC_DPD_MONTH_SHARE', 'OBS_DPD_FLAG', 'mean', 'Tỷ lệ snapshot có DPD > 0 trong DPD hợp lệ', 'tỷ lệ'],
    ['CC_DPD_MAX', 'SK_DPD', 'max', 'DPD lớn nhất theo snapshot', 'ngày'],
    ['CC_BALANCE_MEAN', 'AMT_BALANCE', 'mean', 'Trung bình số dư báo cáo, giữ số âm', 'đơn vị tiền nguồn'],
    ['CC_BALANCE_MAX', 'AMT_BALANCE', 'max', 'Số dư báo cáo lớn nhất', 'đơn vị tiền nguồn'],
    ['CC_LIMIT_MEAN', 'AMT_CREDIT_LIMIT_ACTUAL', 'mean', 'Trung bình hạn mức báo cáo', 'đơn vị tiền nguồn'],
    ['CC_BALANCE_LIMIT_RATIO_MEAN', 'BALANCE_LIMIT_RATIO', 'mean', 'Trung bình tỷ số số dư/hạn mức cùng tài khoản-tháng', 'tỷ số kỹ thuật'],
    ['CC_BALANCE_LIMIT_RATIO_MAX', 'BALANCE_LIMIT_RATIO', 'max', 'Tỷ số số dư/hạn mức lớn nhất với mẫu số dương', 'tỷ số kỹ thuật'],
    ['CC_PAYMENT_TOTAL_MEAN', 'AMT_PAYMENT_TOTAL_CURRENT', 'mean', 'Trung bình tiền thanh toán kỳ báo cáo', 'đơn vị/kỳ nguồn'],
    ['CC_DRAWINGS_MEAN', 'AMT_DRAWINGS_CURRENT', 'mean', 'Trung bình tiền rút kỳ báo cáo', 'đơn vị/kỳ nguồn'],
    ['CC_RECENT_MONTH', 'MONTHS_BALANCE', 'max', 'Tháng tương đối gần nhất có snapshot', 'tháng tương đối']
], columns=['feature', 'source_column', 'aggregation', 'meaning', 'unit'])

card_features['formula'] = card_features['aggregation'] + '(' + card_features['source_column'] + ') per SK_ID_CURR; ratio = balance / limit when limit > 0'
card_features['source_table'] = 'credit_card_balance'
card_features['feature_sets'] = 'B'
card_features['time_rule'] = 'MONTHS_BALANCE < 0; known nonnegative DPD for DPD summaries'
card_features['limitation'] = 'Số dư âm/vượt hạn mức chưa chứng minh lỗi; mean là trung bình snapshot, không phải dư nợ hiện tại'
card_features = card_features.drop(columns='aggregation')
display(card_features[['feature', 'meaning', 'unit']])


,feature,meaning,unit
0,CC_LOAN_N,Số tài khoản có snapshot đủ điều kiện,tài khoản
1,CC_MONTH_N,Số snapshot tài khoản-tháng,snapshot
2,CC_KNOWN_DPD_MONTH_N,Số snapshot có DPD hợp lệ,snapshot
3,CC_DPD_MONTH_SHARE,Tỷ lệ snapshot có DPD > 0 trong DPD hợp lệ,tỷ lệ
4,CC_DPD_MAX,DPD lớn nhất theo snapshot,ngày
5,CC_BALANCE_MEAN,"Trung bình số dư báo cáo, giữ số âm",đơn vị tiền nguồn
6,CC_BALANCE_MAX,Số dư báo cáo lớn nhất,đơn vị tiền nguồn
7,CC_LIMIT_MEAN,Trung bình hạn mức báo cáo,đơn vị tiền nguồn
8,CC_BALANCE_LIMIT_RATIO_MEAN,Trung bình tỷ số số dư/hạn mức cùng tài khoản-...,tỷ số kỹ thuật
9,CC_BALANCE_LIMIT_RATIO_MAX,Tỷ số số dư/hạn mức lớn nhất với mẫu số dương,tỷ số kỹ thuật


In [26]:
aggregate.to_parquet(HISTORY / 'credit_card_balance.parquet')
del aggregate, train_aggregate


Có 3.840.312 snapshot thẻ đủ điều kiện, trong đó 1.941.181 thuộc train. Tỷ số số dư/hạn mức tính cùng tài khoản-tháng trước khi tổng hợp. Số dư âm và tỷ số vượt 1 được giữ; số dư báo cáo chưa được diễn giải thành EAD hoặc tổn thất.

<a id="muc-04-6"></a>

### 04.6. Lịch sử thanh toán: installments_payments
gom các lần thanh toán đã quan sát của kỳ đến hạn theo khóa **SK_ID_CURR, SK_ID_PREV, NUM_INSTALMENT_VERSION, NUM_INSTALMENT_NUMBER**. Kiểm tra nghĩa vụ/ngày hạn thống nhất rồi mới tổng hợp về hồ sơ.

Phiên bản lịch trả 0 có thể là thẻ tín dụng theo từ điển; giữ version trong khóa. Các nhóm khóa là kỳ theo phiên bản lịch được ghi nhận, không mặc định là toàn bộ lịch nghĩa vụ pháp lý. Tỷ lệ tiền đã quan sát/nghĩa vụ chỉ mô tả nhóm có dữ liệu đủ điều kiện. Không suy ra kỳ không có dòng là trả 0, không đặt tên phần chênh lệch là dư nợ quá hạn hiện hữu.

SQL trong `processInstallments()` lọc ngày thanh toán/ngày đến hạn và số tiền, lấy từng khoảng ID theo lô qua `Connector.queryBatches(sql, params)`. pandas kiểm tra ID rồi gom theo hồ sơ/khoản/phiên bản/kỳ, đối chiếu nghĩa vụ và tổng hợp; không coi mỗi dòng thanh toán là một kỳ.


Hàm trả `installment_results` gồm `features`, `checks`, `period_examples` và `payment_examples`. Trong Models, ba bước được viết riêng: gom kỳ hợp lệ → tổng hợp hồ sơ → lấy ví dụ nhỏ ngay trong lô đang đọc. Không truy vấn lại để lấy các ví dụ này.

In [27]:
payment_columns = preparation.payment_columns
period_keys = preparation.period_keys
installment_results = preparation.processInstallments()
aggregate = installment_results['features']


**Xem phạm vi các kỳ trả nợ được sử dụng**

In [28]:
installment_checks = installment_results['checks']
display(installment_checks)


,eligible_payment_rows,usable_versioned_periods,inconsistent_versioned_periods
0,13602251,12948792,0


**Xem một số kỳ có nhiều thanh toán**

In [29]:
period_examples = installment_results['period_examples']
payment_examples = installment_results['payment_examples']
display(period_examples)


,SK_ID_CURR,SK_ID_PREV,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,PAYMENT_ROW_N,OBS_PAID,...,LAST_OBS_PAYMENT,ANY_AFTER_DUE,MAX_OBS_LATE_DAYS,OBS_PAID_SCHEDULED_RATIO,OBS_AMOUNT_COVERED,LAST_OBS_PAYMENT_DAYS_AGO
0,131465,1895324,1.0,6,2,3229.065,...,-1379.0,1,7.0,1.0,0,1379.0
1,197687,1410565,1.0,1,2,12514.050,...,-1029.0,1,8.0,1.0,1,1029.0
2,230433,1118688,1.0,5,2,13788.405,...,-664.0,1,2.0,1.0,1,664.0
3,251099,2044743,1.0,9,3,31026.960,...,-209.0,1,3.0,1.0,1,209.0
4,321324,1780179,0.0,42,2,3375.000,...,-1540.0,1,8.0,1.0,1,1540.0
5,384591,1434236,1.0,11,2,12069.000,...,-2515.0,1,4.0,1.0,1,2515.0
6,419902,2556747,1.0,3,2,2230.155,...,-1653.0,1,2.0,1.0,1,1653.0
7,450479,2192347,0.0,4,2,6750.000,...,-2066.0,1,26.0,1.0,1,2066.0


**Giải thích các đặc trưng trả nợ quan sát được**

In [30]:
definitions = [
    ['INS_OBS_PAYMENT_ROW_N', 'AMT_PAYMENT', 'sum(count rows per versioned period)', 'Số dòng thanh toán đủ điều kiện trong các nhóm kỳ nhất quán', 'dòng'],
    ['INS_OBS_VERSIONED_PERIOD_N', ','.join(period_keys), 'count consistent versioned period keys', 'Số nhóm kỳ theo phiên bản có thanh toán được quan sát', 'nhóm kỳ theo phiên bản'],
    ['INS_OBS_LOAN_N', 'SK_ID_PREV', 'nunique(SK_ID_PREV)', 'Số khoản có nhóm kỳ đủ điều kiện', 'khoản'],
    ['INS_OBS_PAID_SUM', 'AMT_PAYMENT', 'sum observed nonnegative payments in consistent groups', 'Tổng tiền đã quan sát trong phạm vi đủ điều kiện', 'đơn vị tiền nguồn'],
    ['INS_OBS_SCHEDULED_MEAN', 'AMT_INSTALMENT', 'mean one consistent scheduled amount per versioned period', 'Trung bình nghĩa vụ ghi nhận theo nhóm kỳ', 'đơn vị tiền nguồn'],
    ['INS_OBS_PAID_SCHEDULED_RATIO_MEAN', 'AMT_PAYMENT,AMT_INSTALMENT', 'mean(sum observed payments per period / scheduled amount)', 'Trung bình tỷ số tiền quan sát/nghĩa vụ của nhóm kỳ', 'tỷ số kỹ thuật'],
    ['INS_OBS_AMOUNT_COVERED_SHARE', 'AMT_PAYMENT,AMT_INSTALMENT', 'mean(sum observed payment per period >= scheduled amount)', 'Tỷ lệ nhóm kỳ có tiền quan sát đạt nghĩa vụ đã ghi, không yêu cầu đúng hạn', 'tỷ lệ'],
    ['INS_PERIOD_WITH_AFTER_DUE_PAYMENT_SHARE', 'DAYS_ENTRY_PAYMENT,DAYS_INSTALMENT', 'mean(any observed entry > due per versioned period)', 'Tỷ lệ nhóm kỳ có ít nhất một thanh toán sau hạn; chưa chứng minh thiếu tiền tại hạn', 'tỷ lệ'],
    ['INS_MAX_OBS_LATE_DAYS', 'DAYS_ENTRY_PAYMENT,DAYS_INSTALMENT', 'max(max(entry - due, 0))', 'Khoảng trễ lớn nhất của thanh toán được quan sát', 'ngày'],
    ['INS_RECENT_OBS_PAYMENT_DAYS_AGO', 'DAYS_ENTRY_PAYMENT', 'min(-last observed payment day)', 'Số ngày từ thanh toán quan sát gần nhất', 'ngày']
]
installment_features = pd.DataFrame(definitions, columns=['feature', 'source_column', 'formula', 'meaning', 'unit'])
installment_features['source_table'] = 'installments_payments'
installment_features['feature_sets'] = 'B'
installment_features['time_rule'] = 'entry <= 0 and due <= 0; payment >= 0; scheduled > 0; complete versioned keys; consistent due/amount'
installment_features['limitation'] = 'Chỉ nhóm kỳ có dữ liệu đủ điều kiện; các phiên bản chưa chắc là nghĩa vụ kinh tế độc lập; không có toàn bộ lịch nghĩa vụ, không suy ra current arrears/default hoặc trả 0 khi thiếu dòng'



display(installment_features[['feature', 'meaning', 'unit']])


,feature,meaning,unit
0,INS_OBS_PAYMENT_ROW_N,Số dòng thanh toán đủ điều kiện trong các nhóm...,dòng
1,INS_OBS_VERSIONED_PERIOD_N,Số nhóm kỳ theo phiên bản có thanh toán được q...,nhóm kỳ theo phiên bản
2,INS_OBS_LOAN_N,Số khoản có nhóm kỳ đủ điều kiện,khoản
3,INS_OBS_PAID_SUM,Tổng tiền đã quan sát trong phạm vi đủ điều kiện,đơn vị tiền nguồn
4,INS_OBS_SCHEDULED_MEAN,Trung bình nghĩa vụ ghi nhận theo nhóm kỳ,đơn vị tiền nguồn
5,INS_OBS_PAID_SCHEDULED_RATIO_MEAN,Trung bình tỷ số tiền quan sát/nghĩa vụ của nh...,tỷ số kỹ thuật
6,INS_OBS_AMOUNT_COVERED_SHARE,Tỷ lệ nhóm kỳ có tiền quan sát đạt nghĩa vụ đã...,tỷ lệ
7,INS_PERIOD_WITH_AFTER_DUE_PAYMENT_SHARE,Tỷ lệ nhóm kỳ có ít nhất một thanh toán sau hạ...,tỷ lệ
8,INS_MAX_OBS_LATE_DAYS,Khoảng trễ lớn nhất của thanh toán được quan sát,ngày
9,INS_RECENT_OBS_PAYMENT_DAYS_AGO,Số ngày từ thanh toán quan sát gần nhất,ngày


**Lưu bảng thanh toán đã tổng hợp**

In [31]:
aggregate.to_parquet(HISTORY / 'installments_payments.parquet')
del aggregate


13.602.251 dòng đủ điều kiện được gom thành 12.948.792 nhóm kỳ theo phiên bản; không có nhóm lệch ngày hạn/số tiền trong phạm vi đã lọc. Tiền trả được cộng, số tiền theo lịch lấy một lần trong từng nhóm nhất quán. Chưa chứng minh các phiên bản lịch là những nghĩa vụ kinh tế độc lập. Các biến phản ánh thanh toán được quan sát, không phải toàn bộ nghĩa vụ hay dư nợ quá hạn hiện hữu; không có dòng chưa được hiểu là trả 0.

<a id="muc-05"></a>

<a id="part-5"></a>

## 05. Ghép và hoàn thiện bộ B

<a id="muc-05-1"></a>

### 05.1. Ghép lịch sử với bộ A
ghép sáu bảng đã tổng hợp về SK_ID_CURR, giữ nguyên hồ sơ và thứ tự.
`buildModelingDataset()` nhận A, các aggregate đã giữ, danh sách count và cờ nguồn.
Method kiểm tra join một-một, tạo cờ trước khi điền count bằng 0, rồi trả B và bảng đối soát.
Mỗi split được xử lý và xuất một lần; bảng dưới đây cho thấy số dòng và coverage từng nguồn.

**Xác định cờ có lịch sử và các biến đếm**

In [32]:
source_flags = {
    'bureau': ('BUR_LOAN_N', 'HAS_USABLE_BUREAU'),
    'bureau_balance': ('BB_MONTH_N', 'HAS_USABLE_BUREAU_BALANCE'),
    'previous_application': ('PREV_APPLICATION_N', 'HAS_USABLE_PREVIOUS'),
    'pos_cash_balance': ('POS_MONTH_N', 'HAS_USABLE_POS'),
    'credit_card_balance': ('CC_MONTH_N', 'HAS_USABLE_CARD'),
    'installments_payments': ('INS_OBS_VERSIONED_PERIOD_N', 'HAS_USABLE_INSTALLMENTS')
}

count_features = []
count_features.extend(['BUR_LOAN_N', 'BUR_ACTIVE_N', 'BUR_CLOSED_N', 'BUR_OTHER_CURRENCY_N'])
count_features.extend(['BB_LOAN_N', 'BB_MONTH_N', 'BB_KNOWN_DPD_MONTH_N', 'BB_DPD_MONTH_N', 'BB_CLOSED_MONTH_N', 'BB_UNKNOWN_MONTH_N'])
count_features.extend(['PREV_APPLICATION_N', 'PREV_APPROVED_N', 'PREV_REFUSED_N'])
count_features.extend(['POS_LOAN_N', 'POS_MONTH_N', 'POS_KNOWN_DPD_MONTH_N'])
count_features.extend(['CC_LOAN_N', 'CC_MONTH_N', 'CC_KNOWN_DPD_MONTH_N'])
count_features.extend(['INS_OBS_PAYMENT_ROW_N', 'INS_OBS_VERSIONED_PERIOD_N', 'INS_OBS_LOAN_N'])


**Ghép từng tập của A với sáu nguồn lịch sử**

In [33]:
(FEATURES / 'b').mkdir(exist_ok=True)
join_rows = []
coverage_rows = []
for partition in ['train', 'validation', 'test', 'application_test']:
    base = pd.read_parquet(FEATURES / 'a' / (partition + '.parquet'))
    base = base.set_index('SK_ID_CURR')
    base, partition_checks = preparation.buildModelingDataset(base, count_features, source_flags)
    for row in partition_checks.to_dict('records'):
        row['split'] = partition
        join_rows.append(row)
    if partition == 'train':
        for source, (primary_count, flag) in source_flags.items():
            coverage_rows.append({'source': source, 'train_n': len(base),
                'with_usable_n': int(base[flag].sum()),
                'without_usable_n': int((base[flag] == 0).sum())})
    feature_columns_b = base.columns.tolist()
    base.reset_index().to_parquet(FEATURES / 'b' / (partition + '.parquet'), index=False)
    del base
join_checks = pd.DataFrame(join_rows)
join_checks = join_checks[['split', 'source', 'before_n', 'after_n', 'with_usable_source_n']]
display(join_checks.loc[join_checks['split'] == 'train'])
preparation.clearHistoryResults()
del installment_results


,split,source,before_n,after_n,with_usable_source_n
0,train,bureau,184506,184506,158164
1,train,bureau_balance,184506,184506,55257
2,train,previous_application,184506,184506,174636
3,train,pos_cash_balance,184506,184506,173672
4,train,credit_card_balance,184506,184506,52231
5,train,installments_payments,184506,184506,174994


Cả 24 lần ghép (6 nguồn × 4 tập) giữ nguyên số hồ sơ và khóa duy nhất. Sáu nguồn bổ sung 66 đặc trưng lịch sử vào bộ A; các bảng đã được tổng hợp về cấp hồ sơ nên không nhân dòng khi ghép. Ô thiếu sau ghép được đọc cùng cờ có nguồn sử dụng được ở mục sau.

<a id="muc-05-2"></a>

### 05.2. Trạng thái có lịch sử sử dụng được
tạo cờ theo từng nguồn trước khi điền số đếm bằng 0; giữ NaN của thống kê tiền/thời gian khi không có quan sát. Không biến thông tin loại vì sự kiện tương lai thành đặc trưng.

In [34]:
coverage = pd.DataFrame(coverage_rows)
coverage['usable_pct'] = coverage['with_usable_n'] / coverage['train_n'] * 100
display(coverage)


,source,train_n,with_usable_n,without_usable_n,usable_pct
0,bureau,184506,158164,26342,85.722958
1,bureau_balance,184506,55257,129249,29.948620
2,previous_application,184506,174636,9870,94.650580
3,pos_cash_balance,184506,173672,10834,94.128104
4,credit_card_balance,184506,52231,132275,28.308564
5,installments_payments,184506,174994,9512,94.844612


In [35]:
flag_rows = []
time_rule_by_source = time_rules
for source, (primary_count, flag) in source_flags.items():
    flag_rows.append({
        'feature': flag,
        'source_column': primary_count,
        'formula': '1 if eligible aggregate row exists, else 0',
        'meaning': 'Có ít nhất một đơn vị quan sát sử dụng được trong nguồn ' + source,
        'unit': 'cờ',
        'source_table': source,
        'feature_sets': 'B',
        'time_rule': time_rule_by_source[source],
        'limitation': 'Không có nguồn sử dụng được chưa chứng minh người vay mới hoặc lịch sử tốt'
    })
flag_dictionary = pd.DataFrame(flag_rows)
display(flag_dictionary[['feature', 'meaning']])


,feature,meaning
0,HAS_USABLE_BUREAU,Có ít nhất một đơn vị quan sát sử dụng được tr...
1,HAS_USABLE_BUREAU_BALANCE,Có ít nhất một đơn vị quan sát sử dụng được tr...
2,HAS_USABLE_PREVIOUS,Có ít nhất một đơn vị quan sát sử dụng được tr...
3,HAS_USABLE_POS,Có ít nhất một đơn vị quan sát sử dụng được tr...
4,HAS_USABLE_CARD,Có ít nhất một đơn vị quan sát sử dụng được tr...
5,HAS_USABLE_INSTALLMENTS,Có ít nhất một đơn vị quan sát sử dụng được tr...


Thêm 6 cờ, hoàn thành bộ B với 199 đặc trưng. Trên train, độ bao phủ lần lượt là bureau 85,72%, bureau_balance 29,95%, previous 94,65%, POS 94,13%, thẻ 28,31% và installments 94,84%. Chỉ số đếm được điền 0 khi không có quan sát đủ điều kiện; thống kê tiền/thời gian vẫn giữ NaN. Sáu cờ chưa cho biết khách hàng vay lần đầu, lịch sử tốt hoặc mức rủi ro.

<a id="muc-06"></a>

<a id="part-6"></a>

## 06. Lập từ điển đặc trưng
mô tả đầy đủ mỗi biến của A/B bằng nguồn, công thức, đơn vị/kỳ đo, thời điểm, quy tắc thiếu và giới hạn; kiểm tra không có biến thiếu mô tả hoặc trùng tên.

**Mô tả các trường application được giữ lại**

In [ ]:
description_path = ROOT / 'data/metadata/HomeCredit_columns_description.csv'
descriptions = pd.read_csv(description_path, encoding='latin1')
is_application = descriptions['Table'] == 'application_{train|test}.csv'
application_descriptions = descriptions.loc[is_application]
description_map = application_descriptions.set_index('Row')['Description']

original_columns = preparation.original_columns
retained = []
for column in original_columns:
    if column in feature_columns_a:
        retained.append(column)
base_dictionary = pd.DataFrame({'feature': retained, 'source_column': retained})
base_dictionary['meaning'] = base_dictionary['feature'].map(description_map)
base_dictionary['formula'] = 'Giữ giá trị nguồn; categorical NaN -> __MISSING__; OWN_CAR_AGE không có xe -> NaN'
base_dictionary['unit'] = 'Theo từ điển nguồn; kỳ/đơn vị tiền chưa xác minh đầy đủ'
base_dictionary['source_table'] = 'application'
base_dictionary['feature_sets'] = 'A,B'
base_dictionary['time_rule'] = 'Current application information; retrospective availability assumption'
base_dictionary['limitation'] = 'AMT_CREDIT không tự coi là EAD/limit; EXT_SOURCE không tự coi là PD/FICO; ownership không phải đã thế chấp'

**Ghép từ điển đặc trưng để bàn giao**

In [37]:
feature_dictionary = pd.concat([
    derived, base_dictionary, bureau_features, bureau_balance_features,
    previous_features, pos_features, card_features, installment_features, flag_dictionary
], ignore_index=True)
exported_schema = pq.ParquetFile(FEATURES / 'b/train.parquet').schema_arrow
dtype_map = {}
for field in exported_schema:
    dtype_map[field.name] = str(field.type)
feature_dictionary['dtype'] = feature_dictionary['feature'].map(dtype_map)
feature_dictionary['observation_unit'] = 'One current application (SK_ID_CURR)'
feature_dictionary['missing_policy'] = 'Numeric NaN retained: LR/DT/RF fit median/indicator within each fold; CatBoost/XGBoost retain numeric NaN; categorical missing uses fixed token'
feature_dictionary.loc[feature_dictionary['feature'].isin(count_features), 'missing_policy'] = 'No eligible aggregate -> count 0; read with HAS_USABLE source flag'
for flag in source_flags.values():
    feature_dictionary.loc[(feature_dictionary['feature'] == flag[1]), 'missing_policy'] = 'Deterministic 0/1 flag; no imputation'
feature_dictionary.loc[feature_dictionary['feature'].isin(['EMPLOYED_SENTINEL', 'CAR_AGE_NOT_APPLICABLE', 'CAR_AGE_MISSING_WITH_CAR']), 'missing_policy'] = 'Deterministic 0/1 flag; no imputation'
required_metadata = ['feature', 'source_column', 'formula', 'meaning', 'unit', 'source_table',
                     'feature_sets', 'observation_unit', 'dtype', 'time_rule', 'missing_policy', 'limitation']
feature_dictionary = feature_dictionary[required_metadata].sort_values(['feature_sets', 'source_table', 'feature'])
display(feature_dictionary.groupby(['feature_sets', 'source_table']).size().rename('features_n').to_frame())
display(feature_dictionary[['feature', 'meaning', 'unit']].head(12))


features_n
feature_sets source_table                                 
A,B          application                               127
B            bureau                                     15
             bureau_balance                              1
             bureau_balance -> eligible bureau          10
             credit_card_balance                        14
             installments_payments                      11
             pos_cash_balance                           10
             previous_application                       11

,feature,meaning,unit
7,AGE_YEARS,Khoảng thời gian theo cột gốc: DAYS_BIRTH,năm
19,AMT_ANNUITY,Loan annuity,Theo từ điển nguồn; kỳ/đơn vị tiền chưa xác mi...
18,AMT_CREDIT,Credit amount of the loan,Theo từ điển nguồn; kỳ/đơn vị tiền chưa xác mi...
20,AMT_GOODS_PRICE,For consumer loans it is the price of the good...,Theo từ điển nguồn; kỳ/đơn vị tiền chưa xác mi...
17,AMT_INCOME_TOTAL,Income of the client,Theo từ điển nguồn; kỳ/đơn vị tiền chưa xác mi...
122,AMT_REQ_CREDIT_BUREAU_DAY,Number of enquiries to Credit Bureau about the...,Theo từ điển nguồn; kỳ/đơn vị tiền chưa xác mi...
121,AMT_REQ_CREDIT_BUREAU_HOUR,Number of enquiries to Credit Bureau about the...,Theo từ điển nguồn; kỳ/đơn vị tiền chưa xác mi...
124,AMT_REQ_CREDIT_BUREAU_MON,Number of enquiries to Credit Bureau about the...,Theo từ điển nguồn; kỳ/đơn vị tiền chưa xác mi...
125,AMT_REQ_CREDIT_BUREAU_QRT,Number of enquiries to Credit Bureau about the...,Theo từ điển nguồn; kỳ/đơn vị tiền chưa xác mi...
123,AMT_REQ_CREDIT_BUREAU_WEEK,Number of enquiries to Credit Bureau about the...,Theo từ điển nguồn; kỳ/đơn vị tiền chưa xác mi...


Từ điển có 199 dòng tương ứng 199 đặc trưng bộ B; 127 biến dùng chung được đánh dấu A,B. Mỗi dòng có nguồn, công thức, đơn vị, kiểu dữ liệu, quy tắc thiếu và giới hạn. Đơn vị/kỳ đo chưa xác minh được ghi rõ, không tự thay bằng DTI, EAD hoặc PD.

<a id="muc-07"></a>

<a id="part-7"></a>

## 07. Chuẩn bị pipeline tiền xử lý

<a id="muc-07-1"></a>

### 07.1. Đầu vào và cách xử lý từng loại biến
xác định danh sách cột từ schema A/B và chính sách tiền xử lý; loại ID trước khi fit. Cột số dùng median/chỉ báo thiếu trong fold; phân loại dùng token thiếu cố định và OneHot có xử lý mức mới.

In [38]:
schemas = {}
schema_rows = []
for feature_set in ['A', 'B']:
    train_features = pd.read_parquet(FEATURES / feature_set.lower() / 'train.parquet')
    train_features = train_features.set_index('SK_ID_CURR')
    for column in ['TARGET', 'split', 'cv_fold']:
        if column in train_features.columns:
            raise ValueError('Không được đưa nhãn hoặc phân tập vào đặc trưng: ' + column)
    categorical = train_features.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
    numeric = []
    for column in train_features.columns:
        if column not in categorical:
            numeric.append(column)
    schemas[feature_set] = {
        'numeric': numeric,
        'categorical': categorical,
        'feature_order': train_features.columns.tolist()
    }
    schema_rows.append({'set': feature_set, 'numeric_n': len(numeric),
                        'categorical_n': len(categorical), 'features_n': len(train_features.columns)})
    del train_features
display(pd.DataFrame(schema_rows))


,set,numeric_n,categorical_n,features_n
0,A,111,16,127
1,B,183,16,199


A có 111 biến số và 16 biến phân loại; B có 183 biến số và cùng 16 biến phân loại. Schema ghi nhận kiểu và thứ tự cột của bộ A/B; validation/test không được dùng để xác định schema. Cột phân loại thiếu đã có token cố định; bộ mã hóa vẫn chỉ học các mức trong fold train.

<a id="muc-07-2"></a>

### 07.2. Nhánh Logistic Regression và nhánh cây
chuẩn bị hai cách tiền xử lý cho mỗi bộ A/B: nhánh LR có scaling biến số, nhánh cây không scaling. Giữ cột toàn thiếu trong fold và dữ liệu dạng sparse để hạn chế RAM.

Scaling là lựa chọn hỗ trợ solver/regularization của LR trong thiết kế này, không phải điều kiện toán học bắt buộc cho mọi LR. Chưa loại nhóm OneHot đầu tiên; các mô hình regularized sẽ được thiết kế ở notebook 06.

In [39]:
preprocessor_templates = {}
for feature_set, schema in schemas.items():
    preprocessor_templates[feature_set + '_LR'] = preparation.makePreprocessor(schema, True)
    preprocessor_templates[feature_set + '_TREE'] = preparation.makePreprocessor(schema, False)

print('Các bộ tiền xử lý chưa fit:', list(preprocessor_templates))


Có bốn template cho A/B và hai nhánh. Các đối tượng chưa chứa tham số đã học; notebook 06 phải tạo bản mới và fit trong từng fold, không dùng lại transformer đã fit thử ở mục dưới. Chưa thêm classifier vào pipeline.

<a id="muc-07-3"></a>

### 07.3. Xem kết quả tiền xử lý theo CV fold
minh họa việc fit trên hai fold train và transform fold còn lại. Bảng cho biết số quan sát và số chiều sau biến đổi ở A/B, LR/cây; chưa fit classifier.

In [40]:
pipeline_rows = []
fold_by_id = split.set_index('SK_ID_CURR')['cv_fold']
for feature_set in ['A', 'B']:
    train_features = pd.read_parquet(FEATURES / feature_set.lower() / 'train.parquet')
    train_features = train_features.set_index('SK_ID_CURR')
    folds = fold_by_id.loc[train_features.index]
    for branch in ['LR', 'TREE']:
        for fold in [0, 1, 2]:
            is_fit = folds != fold
            is_probe = folds == fold
            template = preprocessor_templates[feature_set + '_' + branch]
            transformer = clone(template)
            transformer.fit(train_features.loc[is_fit])
            transformed = transformer.transform(train_features.loc[is_probe])
            pipeline_rows.append({
                'set': feature_set, 'branch': branch, 'fold': fold,
                'fit_n': int(is_fit.sum()), 'probe_n': int(is_probe.sum()),
                'output_columns_n': transformed.shape[1]
            })
            del transformer, transformed
    del train_features
pipeline_summary = pd.DataFrame(pipeline_rows)
display(pipeline_summary)


,set,branch,fold,fit_n,probe_n,output_columns_n
0,A,LR,0,123004,61502,318
1,A,LR,1,123004,61502,322
2,A,LR,2,123004,61502,321
3,A,TREE,0,123004,61502,318
4,A,TREE,1,123004,61502,322
5,A,TREE,2,123004,61502,321
6,B,LR,0,123004,61502,434
7,B,LR,1,123004,61502,438
8,B,LR,2,123004,61502,437
9,B,TREE,0,123004,61502,434


Mỗi lượt fit 123.004 hồ sơ và transform 61.502 hồ sơ trong train. A có 318–322 cột sau biến đổi, B có 434–438 cột tùy fold. Số chiều khác nhau do mức phân loại/cờ thiếu được học từ phần fit. notebook 06 sẽ tạo pipeline mới cho mỗi lượt CV; không tái sử dụng các transformer fit thử ở đây.

<a id="muc-08"></a>

<a id="part-8"></a>

## 08. Đọc đầu ra và đối chiếu ý nghĩa đặc trưng
nhìn quy mô A/B theo phân tập, rồi xem trực tiếp một kỳ có nhiều thanh toán và một hồ sơ có nhiều mã tiền. Đây là ví dụ giải thích công thức, không phải quy tắc chọn thêm đặc trưng.

In [41]:
dataset_rows = []
for partition in ['train', 'validation', 'test', 'application_test']:
    a_file = pq.ParquetFile(FEATURES / 'a' / (partition + '.parquet'))
    b_file = pq.ParquetFile(FEATURES / 'b' / (partition + '.parquet'))
    dataset_rows.append({
        'split': partition,
        'A_rows': a_file.metadata.num_rows,
        'B_rows': b_file.metadata.num_rows,
        'A_features_n': len(a_file.schema_arrow) - 1,
        'B_features_n': len(b_file.schema_arrow) - 1
    })
display(pd.DataFrame(dataset_rows))


,split,A_rows,B_rows,A_features_n,B_features_n
0,train,184506,184506,127,199
1,validation,61502,61502,127,199
2,test,61503,61503,127,199
3,application_test,48744,48744,127,199


**Đọc từng lần trả tiền trong một kỳ minh họa**

In [42]:
example = period_examples.iloc[0]
raw_payments = preparation.readPaymentPeriod(example)
known_dates = (raw_payments['DAYS_ENTRY_PAYMENT'] <= 0) & (raw_payments['DAYS_INSTALMENT'] <= 0)
valid_amounts = (raw_payments['AMT_PAYMENT'] >= 0) & (raw_payments['AMT_INSTALMENT'] > 0)
observed_payments = raw_payments.loc[known_dates & valid_amounts]
display(observed_payments)

payment_comparison = pd.DataFrame({
    'measure': ['Payment rows', 'Observed paid', 'Scheduled amount'],
    'source_rows': [len(observed_payments), observed_payments['AMT_PAYMENT'].sum(),
                    observed_payments['AMT_INSTALMENT'].iloc[0]],
    'period_summary': [example['PAYMENT_ROW_N'], example['OBS_PAID'], example['SCHEDULED_MIN']]
})
display(payment_comparison)


,SK_ID_CURR,SK_ID_PREV,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,131465,1895324,1.0,6,-1386.0,-1379.0,3229.065,3212.865
1,131465,1895324,1.0,6,-1386.0,-1413.0,3229.065,16.200


,measure,source_rows,period_summary
0,Payment rows,2.000,2.000
1,Observed paid,3229.065,3229.065
2,Scheduled amount,3229.065,3229.065


**Minh họa vì sao không cộng tiền khác currency**

In [43]:
currency_example_id = preparation.currency_example_id
currency_source = preparation.readBureauApplication(currency_example_id)
known_dates = (currency_source['DAYS_CREDIT'] <= 0) & (currency_source['DAYS_CREDIT_UPDATE'] <= 0)
ordered_dates = currency_source['DAYS_CREDIT_UPDATE'] >= currency_source['DAYS_CREDIT']
is_application = currency_source['SK_ID_CURR'] == currency_example_id
currency_example = currency_source.loc[known_dates & ordered_dates & is_application]
display(currency_example[['SK_ID_CURR', 'CREDIT_CURRENCY', 'AMT_CREDIT_SUM']])

is_currency_one = currency_example['CREDIT_CURRENCY'] == 'currency 1'
currency_one_credit = currency_example.loc[is_currency_one, 'AMT_CREDIT_SUM'].mean()
bureau_output = pd.read_parquet(HISTORY / 'bureau.parquet').loc[currency_example_id]
currency_comparison = pd.DataFrame({
    'measure': ['Mean credit in currency 1', 'Other currency records'],
    'source_rows': [currency_one_credit, int((~is_currency_one).sum())],
    'feature_value': [bureau_output['BUR_C1_CREDIT_MEAN'], bureau_output['BUR_OTHER_CURRENCY_N']]
})
display(currency_comparison)
del currency_source, currency_example, bureau_output


,SK_ID_CURR,CREDIT_CURRENCY,AMT_CREDIT_SUM
0,253537,currency 1,1.350000e+07
1,253537,currency 2,4.737420e+06
2,253537,currency 1,0.000000e+00
3,253537,currency 1,3.645000e+05
4,253537,currency 1,6.750000e+05
5,253537,currency 1,3.060000e+05
6,253537,currency 1,9.000000e+05
7,253537,currency 1,3.988324e+06
8,253537,currency 1,1.350000e+06
9,253537,currency 1,0.000000e+00


,measure,source_rows,feature_value
0,Mean credit in currency 1,2.136732e+06,2.136732e+06
1,Other currency records,1.000000e+00,1.000000e+00


In [44]:
history_scope = pd.DataFrame(preparation.scope_rows)
display(history_scope)


,source,source_rows,eligible_rows,train_eligible_rows
0,bureau,1716428,1716276,879778
1,bureau_balance,27299925,23590930,8663845
2,previous_application,1670214,1670214,848478
3,pos_cash_balance,10001358,10001358,5125192
4,credit_card_balance,3840312,3840312,1941181
5,installments_payments,13605401,13602251,6957607


Bộ A/B giữ một dòng cho mỗi hồ sơ. Nghĩa vụ của kỳ chỉ được ghi một lần khi cộng các lần thanh toán; tiền bureau chỉ được tổng hợp trong currency 1. Các bảng phạm vi cho biết dữ liệu nào đủ điều kiện, nhưng chưa xác minh được toàn bộ thời điểm sẵn có trong thực tế.

<a id="muc-09"></a>

<a id="part-9"></a>

## 09. Xuất kết quả và chốt bước chuẩn bị dữ liệu
bàn giao đặc trưng A/B, chỉ mục phân tập, từ điển và schema. Chỉ schema cột là đầu vào tiền xử lý của notebook 06; các quyết định nghiệp vụ đã được giải thích trong notebook và từ điển.

In [ ]:
feature_index.to_parquet(FEATURES / 'feature_index.parquet', index=False)
feature_dictionary.to_csv(ROOT / 'data/metadata/feature_dictionary.csv', index=False)

# Notebook 06 cần các nhóm cột và thứ tự, không dùng transformer đã fit thử.
preprocessing_schema = {'schemas': schemas}
schema_path = ROOT / 'configs/preprocessing_schema.json'
schema_text = json.dumps(preprocessing_schema, ensure_ascii=False, indent=2)
schema_path.write_text(schema_text, encoding='utf-8')
print('Đã lưu chỉ mục phân tập, từ điển và schema cho notebook 06.')


In [46]:
connector.disConnect()


Hoàn thành bộ A gồm 127 đặc trưng và bộ B gồm 199 đặc trưng cho bốn phân tập, cùng sáu bảng lịch sử ở cấp hồ sơ. Các công thức tạo biến không dùng TARGET; số thiếu được giữ để từng họ mô hình xử lý đúng trong bước huấn luyện.

Notebook 05 chọn biến từ bộ B. Notebook 06 đọc đặc trưng qua MySQL và chạy CV sau Feature Selection với 181 biến/fold. Notebook 07 đã fit bản cuối trên toàn train với 182 biến, kiểm tra calibration và chọn ngưỡng; notebook 08 giải thích đúng artifact mới.

Các transformer fit thử tại notebook này chỉ phục vụ kiểm tra kỹ thuật và không được đưa sang bước lựa chọn biến hoặc huấn luyện mô hình.
